# pf_helper: GridKit power flow solver
* conda: h-py312-basic
* notes and design details: [`work-notes/pf_helper.md`](../work-notes/pf_helper.md)

Run GridKit's Newton/KINSOL PF solver (`solve_pf`) on MATPOWER `.m` cases.
`solve_pf` is a thin wrapper around GridKit's `PowerFlow` module — built once from
`uq-usecase/pf-solver/` (Section 2), then called via `subprocess` for each case.

## workflow
1. **Section 1**: run `grid3bus` (built-in 3-bus demo), verify expected solution
2. **Section 2**: build `solve_pf` from `uq-usecase/pf-solver/` (one-time)
3. **Section 3**: sanity check `solve_pf` on `3bus.mat`, compare to `grid3bus` reference
4. **Section 4**: run `solve_pf` on `case_ACTIVSg200.m` (200-bus Illinois)
5. **Section 5**: run `solve_pf` on `Hawaii40_20231026.m` (37-bus Hawaii)


In [3]:
import os
import re
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = "all"
pd.set_option("display.max_rows", 20)
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", "{:.6f}".format)

onkestrel = "NREL_CLUSTER" in os.environ and os.environ["NREL_CLUSTER"] == "kestrel"

# === GridKit paths ===
GRIDKIT_REPO = Path.home() / "gridkit"
BUILD_DIR = GRIDKIT_REPO / "build"
UQ_DIR = GRIDKIT_REPO / "uq-usecase"
PF_SRC_DIR = UQ_DIR / "pf-solver"
PF_BUILD_DIR = PF_SRC_DIR / "build"  # local build dir for solve_pf only

GRID3BUS_BIN = BUILD_DIR / "examples/PowerFlow/Grid3Bus/grid3bus"
SOLVE_PF_BIN = PF_BUILD_DIR / "solve_pf"
BUS3_MAT = GRIDKIT_REPO / "examples/PowerFlow/Grid3Bus/3bus.mat"

# === case data ===
SCIDAC_DATA = Path("/kfs2/projects/scidac/scidac-data")

# Illinois (ACTIVSg200, 200-bus)
ILLINOIS_M = SCIDAC_DATA / "ACTIVSg200/raw-tamu-data/case_ACTIVSg200.m"

# Hawaii (Hawaii40, 37-bus)
HAWAII_M = SCIDAC_DATA / "Hawaii40/raw-tamu-data/Hawaii40_20231026.m"

for label, p in [
    ("grid3bus binary", GRID3BUS_BIN),
    ("solve_pf binary", SOLVE_PF_BIN),
    ("3bus.mat", BUS3_MAT),
    ("Illinois .m", ILLINOIS_M),
    ("Hawaii .m", HAWAII_M),
    ("solve_pf src", PF_SRC_DIR / "solve_pf.cpp"),
]:
    status = "OK " if p.exists() else "MISSING"
    print(f"  [{status}]  {label}: {p}")

  [OK ]  grid3bus binary: /home/isatkaus/gridkit/build/examples/PowerFlow/Grid3Bus/grid3bus
  [MISSING]  solve_pf binary: /home/isatkaus/gridkit/uq-usecase/pf-solver/build/solve_pf
  [OK ]  3bus.mat: /home/isatkaus/gridkit/examples/PowerFlow/Grid3Bus/3bus.mat
  [OK ]  Illinois .m: /kfs2/projects/scidac/scidac-data/ACTIVSg200/raw-tamu-data/case_ACTIVSg200.m
  [OK ]  Hawaii .m: /kfs2/projects/scidac/scidac-data/Hawaii40/raw-tamu-data/Hawaii40_20231026.m
  [OK ]  solve_pf src: /home/isatkaus/gridkit/uq-usecase/pf-solver/solve_pf.cpp


# Section 1: 3-bus demo via `grid3bus` binary

The `grid3bus` binary is already built as part of the main GridKit CMake build.
It runs three variants of the same 3-bus problem (monolithic, parser, hardwired)
and prints results to stdout. The `.m` data is baked in — no file argument needed.

Expected solution: `theta2 = -4.87979 deg`, `V2 = 1.08281 p.u.`, `theta3 = 1.46241 deg`


In [4]:
result = subprocess.run([str(GRID3BUS_BIN)], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
    print(f"Exit code: {result.returncode}")

--------------------------------

Solving power flow for a 3-bus monolithic model ...

Model size: 3

Solution:
  theta2 = -4.8798 deg,  expected = -4.87979 deg
  V2     = 1.08281 p.u., expected = 1.08281 p.u.
  theta3 = 1.46241 deg,  expected = 1.46241 deg

Nonlinear iters               = 9
Nonlinear fn evals            = 10
Beta condition fails          = 0
Backtrack operations          = 0
Nonlinear fn norm             = 7.10879756366045e-06
Step length                   = 1.25128228043541e-06
Jac fn evals                  = 1
LS Nonlinear fn evals         = 3
Prec setup evals              = 0
Prec solves                   = 0
LS iters                      = 0
LS fails                      = 0
Jac-times evals               = 0
LS iters per NLS iter         = 0
Jac evals per NLS iter        = 0.111111111111111
Prec evals per NLS iter       = 0

Success!


--------------------------------
Solving same problem, but assembled from components via a parser ...

Model size: 3

Solution:
  

In [6]:
def parse_grid3bus_output(stdout: str) -> pd.DataFrame:
    """Parse theta/V lines from grid3bus stdout into a small DataFrame."""
    rows = []
    current_case = None
    for line in stdout.splitlines():
        ll = line.lower()
        if "monolithic" in ll:
            current_case = "monolithic"
        elif "via a parser" in ll:
            current_case = "parser"
        elif "manually" in ll:
            current_case = "hardwired"
        m = re.match(r"\s*theta(\d+)\s*=\s*([-\d.]+)\s*deg", line)
        if m:
            rows.append(
                {
                    "case": current_case,
                    "var": f"theta{m.group(1)} (deg)",
                    "value": float(m.group(2)),
                }
            )
        m = re.match(r"\s*V(\d+)\s*=\s*([-\d.]+)\s*p\.u\.", line)
        if m:
            rows.append(
                {
                    "case": current_case,
                    "var": f"V{m.group(1)} (p.u.)",
                    "value": float(m.group(2)),
                }
            )
    return pd.DataFrame(rows).pivot(index="var", columns="case", values="value")


df_3bus = parse_grid3bus_output(result.stdout)
print("3-bus solution (all three variants):")
df_3bus

3-bus solution (all three variants):


case,hardwired,monolithic,parser
var,,,
V2 (p.u.),1.082810,1.082810,1.082810
theta2 (deg),-4.879800,-4.879800,-4.879800
theta3 (deg),1.462410,1.462410,1.462400


# Section 2: build `solve_pf` (one-time)

`solve_pf` (`uq-usecase/pf-solver/solve_pf.cpp`) is a thin C++ wrapper around GridKit's
`PowerFlow` module. Build it once with:

```bash
bash ~/gridkit/uq-usecase/pf-solver/build.sh
```

The script loads the same modules and clang used for the main GridKit build, then compiles
against `~/gridkit/build/` (no changes to the main build tree). Output:
`~/gridkit/uq-usecase/pf-solver/build/solve_pf`

After building, `solve_pf` accepts any MATPOWER `.m` or `.mat` file as `argv[1]` and prints:
```
bus <i>  V=<pu>  theta_deg=<deg>  type=<1|2|3>
```
Bus results go to stdout; solver diagnostics (KINSOL stats) go to stderr.


In [11]:
# Check that solve_pf binary exists before proceeding.
# If MISSING, run the build script once in a terminal:
#   bash ~/gridkit/uq-usecase/pf-solver/build.sh
if SOLVE_PF_BIN.exists():
    print(f"[OK]  solve_pf binary found: {SOLVE_PF_BIN}")
else:
    print(f"[MISSING]  {SOLVE_PF_BIN}")
    print()
    print("Build it with:")
    print(f"  bash {PF_SRC_DIR}/build.sh")

[OK]  solve_pf binary found: /home/isatkaus/gridkit/uq-usecase/pf-solver/build/solve_pf


# Section 3: sanity check solve_pf on 3-bus case

Run `solve_pf` on `3bus.mat` and compare to the `grid3bus` reference values.

Expected: `theta2 = -4.87979 deg`, `V2 = 1.08281 p.u.`, `theta3 = 1.46241 deg`

Note: `solve_pf` prints bus results to **stdout** and solver diagnostics (parse counts,
KINSOL stats) to **stderr** — they are captured separately below.


In [15]:
def run_solve_pf(m_path: Path) -> tuple[pd.DataFrame, str, int]:
    """
    Run solve_pf on a .m file.
    Returns (bus_df, stderr_text, return_code).
    bus_df columns: bus_i, V_pu, theta_deg, type
    """
    r = subprocess.run([str(SOLVE_PF_BIN), str(m_path)], capture_output=True, text=True)
    rows = []
    for line in r.stdout.splitlines():
        # format: bus <i>  V=<v>  theta_deg=<t>  type=<t>
        m = re.match(
            r"bus\s+(\d+)\s+V=([\d.eE+\-]+)\s+theta_deg=([\d.eE+\-]+)\s+type=(\d+)",
            line.strip(),
        )
        if m:
            rows.append(
                {
                    "bus_i": int(m.group(1)),
                    "V_pu": float(m.group(2)),
                    "theta_deg": float(m.group(3)),
                    "type": int(m.group(4)),
                }
            )
    df = pd.DataFrame(rows)
    return df, r.stderr, r.returncode


r3_df, r3_stderr, r3_rc = run_solve_pf(BUS3_MAT)
print("STDERR (solver diagnostics):")
print(r3_stderr)
print(f"Return code: {r3_rc}")

STDERR (solver diagnostics):
Reading: /home/isatkaus/gridkit/examples/PowerFlow/Grid3Bus/3bus.mat
Parsed: 3 buses, 2 gens, 3 branches
Model size (DOF): 3
KINSOL return code: 0

Return code: 0


In [16]:
ref_3bus = {
    "theta2_deg": -4.87979,
    "V2_pu": 1.08281,
    "theta3_deg": 1.46241,
}

print("solve_pf output:")
print(r3_df.to_string(index=False))
print()

if not r3_df.empty:
    bus2 = r3_df[r3_df.bus_i == 2].iloc[0]
    bus3 = r3_df[r3_df.bus_i == 3].iloc[0]
    tol = 1e-3
    checks = {
        "theta2 (deg)": (bus2.theta_deg, ref_3bus["theta2_deg"]),
        "V2 (p.u.)": (bus2.V_pu, ref_3bus["V2_pu"]),
        "theta3 (deg)": (bus3.theta_deg, ref_3bus["theta3_deg"]),
    }
    print(f"{'var':<16} {'got':>12} {'expected':>12} {'|err|':>10} {'pass?':>6}")
    for var, (got, exp) in checks.items():
        err = abs(got - exp)
        ok = "PASS" if err < tol else "FAIL"
        print(f"{var:<16} {got:>12.5f} {exp:>12.5f} {err:>10.2e} {ok:>6}")

solve_pf output:
 bus_i     V_pu  theta_deg  type
     1 1.000000   0.000000     3
     2 1.082809  -4.879799     1
     3 1.100000   1.462404     2

var                       got     expected      |err|  pass?
theta2 (deg)         -4.87980     -4.87979   9.41e-06   PASS
V2 (p.u.)             1.08281      1.08281   1.26e-06   PASS
theta3 (deg)          1.46240      1.46241   6.17e-06   PASS


# Section 4: run solve_pf on ACTIVSg200 (200-bus Illinois)

`solve_pf` reads `case_ACTIVSg200.m` (200 buses, 245 branches, 49 generators) and
runs the GridKit KINSOL solver, warm-started from the Vm/Va values already in the `.m`
file (the TAMU base-case converged PF solution).

**Return code 0** = converged (or stagnated at warm-start point, which means
the initial point already satisfies the residual tolerance).

After a successful solve, compare the resulting Vm/Va to the `.m` reference values.
They should agree closely since we warm-started from that same operating point.


In [21]:
il_df, il_stderr, il_rc = run_solve_pf(ILLINOIS_M)
print("STDERR (solver diagnostics):")
print(il_stderr)
print(f"\nReturn code: {il_rc}")
print(f"Buses parsed from stdout: {len(il_df)}")

STDERR (solver diagnostics):
Reading: /kfs2/projects/scidac/scidac-data/ACTIVSg200/raw-tamu-data/case_ACTIVSg200.m
Parsed (local parser): baseMVA=100  buses=200  gens=49  branches=245
Parsed: 200 buses, 49 gens, 245 branches
Model size (DOF): 350
[ERROR][rank 0][/nopt/nrel/apps/cpu_stack/software/gridkit/src/sundials-src/src/kinsol/kinsol.c:747][KINSol] The line search algorithm was unable to find an iterate sufficiently distinct from the current iterate.

ERROR: Function KINSol failed with flag -5!

terminate called after throwing an instance of 'AnalysisManager::Sundials::SundialsException'
  what():  Method in Kinsol class failed!



Return code: -6
Buses parsed from stdout: 0


In [ ]:
if il_rc != 0:
    print("solve_pf DID NOT CONVERGE (or failed to run). Check stderr above.")
else:
    print("solve_pf converged!\n")
    print("Bus voltage summary:")
    print(il_df[["bus_i", "V_pu", "theta_deg", "type"]].describe())
    print()
    print("First 10 buses:")
    il_df.head(10)

In [ ]:
if il_rc == 0 and not il_df.empty:
    # read reference VM/VA from the .m file
    import re as _re

    with open(ILLINOIS_M) as f:
        content = f.read()
    bus_block = _re.search(r"mpc\.bus\s*=\s*\[(.*?)\];", content, _re.DOTALL).group(1)
    ref_rows = []
    for line in bus_block.strip().splitlines():
        cols = line.strip().rstrip(";").split()
        if not cols or cols[0].startswith("%"):
            continue
        ref_rows.append(
            {
                "bus_i": int(cols[0]),
                "VM_ref": float(cols[7]),
                "VA_ref": float(cols[8]),
            }
        )
    ref_df = pd.DataFrame(ref_rows)

    cmp = il_df.merge(ref_df, on="bus_i")
    cmp["V_err"] = (cmp.V_pu - cmp.VM_ref).abs()
    cmp["theta_err"] = (cmp.theta_deg - cmp.VA_ref).abs()

    print("Comparison: solve_pf result vs .m reference VM/VA")
    print(f"  max |V err|     = {cmp.V_err.max():.6f} pu")
    print(f"  mean |V err|    = {cmp.V_err.mean():.6f} pu")
    print(f"  max |theta err| = {cmp.theta_err.max():.6f} deg")
    print(f"  mean |theta err|= {cmp.theta_err.mean():.6f} deg")
    print()
    print("Sample (first 10 buses):")
    cmp[["bus_i", "V_pu", "VM_ref", "V_err", "theta_deg", "VA_ref", "theta_err"]].head(
        10
    )

# Section 5: run solve_pf on Hawaii40 (37-bus Hawaii)

`Hawaii40_20231026.m` — 37 buses, 89 branches, 45 generators (39 synchronous, 6 offline).
This is the MATPOWER source for `hawaii.json`. A successful PF solve here gives the
initial conditions (Vr/Vi per bus, p0/q0 per gen) that would be needed if we want to
vary load/dispatch and re-solve before patching `hawaii.json` for dynamic simulation.

**Return code 0** = converged. After convergence, compare Vm/Va against the values already
stored in the `.m` file (columns 8/9 of `mpc.bus`), which are a prior PF solution.


In [ ]:
hw_df, hw_stderr, hw_rc = run_solve_pf(HAWAII_M)
print("STDERR (solver diagnostics):")
print(hw_stderr)
print(f"\nReturn code: {hw_rc}")
print(f"Buses parsed from stdout: {len(hw_df)}")

In [ ]:
if hw_rc != 0:
    print("solve_pf DID NOT CONVERGE (or failed to run). Check stderr above.")
else:
    print("solve_pf converged!\n")
    print("Bus voltage summary:")
    print(hw_df[["bus_i", "V_pu", "theta_deg", "type"]].describe())
    print()
    print("All buses:")
    hw_df

In [ ]:
if hw_rc == 0 and not hw_df.empty:
    # read reference VM/VA from the .m file (columns 8/9 of mpc.bus, 0-based: indices 7, 8)
    with open(HAWAII_M) as f:
        content = f.read()
    bus_block = re.search(r"mpc\.bus\s*=\s*\[(.*?)\];", content, re.DOTALL).group(1)
    ref_rows = []
    for line in bus_block.strip().splitlines():
        cols = line.strip().rstrip(";").split()
        if not cols or cols[0].startswith("%"):
            continue
        ref_rows.append(
            {
                "bus_i": int(float(cols[0])),
                "VM_ref": float(cols[7]),
                "VA_ref": float(cols[8]),
            }
        )
    ref_df = pd.DataFrame(ref_rows)

    cmp = hw_df.merge(ref_df, on="bus_i")
    cmp["V_err"] = (cmp.V_pu - cmp.VM_ref).abs()
    cmp["theta_err"] = (cmp.theta_deg - cmp.VA_ref).abs()

    print("Comparison: solve_pf result vs .m reference VM/VA")
    print(f"  max |V err|      = {cmp.V_err.max():.6f} pu")
    print(f"  mean |V err|     = {cmp.V_err.mean():.6f} pu")
    print(f"  max |theta err|  = {cmp.theta_err.max():.6f} deg")
    print(f"  mean |theta err| = {cmp.theta_err.mean():.6f} deg")
    print()
    cmp[["bus_i", "V_pu", "VM_ref", "V_err", "theta_deg", "VA_ref", "theta_err"]]